# Shape Functions on 5-node pyramid

This notebook derives the shape functions for a 5-node 3D finite element (pyramid).
Shape functions $N_i$ are used to interpolate values within an element based on nodal values.
Note that for a pyramid, standard polynomial shape functions that are linear on all edges are often rational.
However, for a simple pyramid with a square base at $z=0$ and an apex at $z=1$, we can use certain basis functions.


In [ ]:
from sympy import symbols, Eq, solve, simplify, IndexedBase, diff


In [ ]:
from typst_utils import typst_print

## Symbol Definition
We define the 3D position vector $\mathbf{X} = [\xi, \eta, \zeta]^T$ and the positions of the five nodes $\mathbf{X}_i$ for $i=0, \dots, 4$.


In [ ]:
# Define symbols
X = IndexedBase('referenceposition')
# Using IndexedBase for each node to represent vectors
nodes = [IndexedBase(f'referenceposition^(({i}))') for i in range(5)]


## Combination
We assume that each shape function $N_i$ is a linear combination of the following terms:
$\{1, X[0], X[1], X[2], X[0]X[1], X[0]X[1]X[2]\}$
Wait, for 5 nodes we only need 5 basis functions.
Standard pyramid shape functions are often defined using $(1-\zeta)$ as a factor for the base.

For simplicity and following the pattern, let's try to solve for 5 coefficients using basis:
$\{1, X[0], X[1], X[2], X[0]X[1]\}$


In [ ]:
# Define symbols for coefficients
coeffs = symbols('a b c d e')


## Solving the System
The shape functions must satisfy the Kronecker delta property: $N_i(\mathbf{X}_j) = \delta_{ij}$.


In [ ]:
# Solve for each shape function N[i]
N = []
for i in range(5):
    # Ni(X) = a + b*X0 + c*X1 + d*X2 + e*X0*X1
    eqs = [
        Eq(coeffs[0] + 
           coeffs[1] * nodes[j][0] + 
           coeffs[2] * nodes[j][1] + 
           coeffs[3] * nodes[j][2] + 
           coeffs[4] * nodes[j][0] * nodes[j][1], 1 if i == j else 0)
        for j in range(5)
    ]
    sol = solve(eqs, coeffs)
    
    Ni = (sol[coeffs[0]] + 
          sol[coeffs[1]] * X[0] + 
          sol[coeffs[2]] * X[1] + 
          sol[coeffs[3]] * X[2] + 
          sol[coeffs[4]] * X[0] * X[1])
    N.append(Ni.simplify())


## Symbolic Results
The general symbolic expressions for the shape functions are:


In [ ]:
# Print results
for i in range(5):
    print(f"Shape function N[{i}]:\n{N[i]}\n")


## Symbolic Gradients
The symbolic gradients of the shape functions with respect to $\xi$, $\eta$ and $\zeta$ are:


In [ ]:
# Compute gradients
dN_dxi = [diff(Ni, X[0]) for Ni in N]
dN_deta = [diff(Ni, X[1]) for Ni in N]
dN_dzeta = [diff(Ni, X[2]) for Ni in N]

# Print gradients
for i in range(5):
    print(f"Gradient dN[{i}]/dxi: {dN_dxi[i]}")
    print(f"Gradient dN[{i}]/deta: {dN_deta[i]}")
    print(f"Gradient dN[{i}]/dzeta: {dN_dzeta[i]}")
    print()


## Numerical Evaluation
We can specialize these general expressions for specific reference elements.


### Standard Reference Pyramid
Base at $z=0$ with nodes (0,0,0), (1,0,0), (1,1,0), (0,1,0) and apex at (0,0,1).
Wait, the apex is usually at (0.5, 0.5, 1) or (0,0,1) depending on convention.
Let's use (0,0,0), (1,0,0), (1,1,0), (0,1,0) for the base and (0,0,1) for the apex.


In [ ]:
# Standard reference pyramid values (5 nodes)
vals_std = {
    nodes[0][0]: 0, nodes[0][1]: 0, nodes[0][2]: 0,
    nodes[1][0]: 1, nodes[1][1]: 0, nodes[1][2]: 0,
    nodes[2][0]: 1, nodes[2][1]: 1, nodes[2][2]: 0,
    nodes[3][0]: 0, nodes[3][1]: 1, nodes[3][2]: 0,
    nodes[4][0]: 0, nodes[4][1]: 0, nodes[4][2]: 1
}

N_numeric_std = [Ni.subs(vals_std) for Ni in N]
dN_dxi_numeric_std = [dNi.subs(vals_std) for dNi in dN_dxi]
dN_deta_numeric_std = [dNi.subs(vals_std) for dNi in dN_deta]
dN_dzeta_numeric_std = [dNi.subs(vals_std) for dNi in dN_dzeta]

for i in range(5):
    print(f"Shape function N[{i}] (standard): {N_numeric_std[i].simplify()}")
    print(f"Gradient dN[{i}]/dxi (standard): {dN_dxi_numeric_std[i]}")
    print(f"Gradient dN[{i}]/deta (standard): {dN_deta_numeric_std[i]}")
    print(f"Gradient dN[{i}]/dzeta (standard): {dN_dzeta_numeric_std[i]}")
    print()
